# 05.01 — Introducing the GraphProfile

Orthograph has two complementary representations of a graph:

| | **GraphDefinition** | **GraphProfile** |
|---|---|---|
| Nature | Declared intent | Observed reality |
| Built from | Python model classes | Database inspection |
| Contains | Labels, types, properties, cardinality bounds | Counts, types found, completeness, distributions |
| Used for | Schema declaration, Cypher validation | Drift detection, data-quality checks |

A `GraphProfile` is a **structural snapshot** of what actually exists in a graph
database at a point in time.  It records, for each node label and relationship
type:

- how many instances were observed,
- which properties were present and at what completeness (ADR-034 §5),
- which value types were observed per property,
- optional cardinality distributions (degree statistics), and
- optional per-partition breakdowns for conditional relationships (E41).

This notebook shows how to **construct** a `GraphProfile` by hand — useful for
learning the data model and for writing tests — and then introduces the main
fields through a series of short examples.  Live-database inspection (using
`orthograph.api.database.inspect`) is covered in 04.01–04.02; comparison
against a definition is covered in 05.02.

Sections:
1. The minimal profile — just counts
2. Adding property profiles — completeness and types
3. `BoundedDistribution` — the shared statistical primitive (ADR-034 §3)
4. Cardinality statistics on relationships
5. Constraint cross-reference — `constraint_required` (ADR-034 §4)
6. Value distributions and truncation honesty
7. Serialising and reloading a profile (JSON round-trip)
8. Rendering a profile as text

In [1]:
from orthograph.api import visualization
from orthograph.graph_profile.models import (
    BoundedDistribution,
    CardinalityStats,
    ConstraintInfo,
    GraphProfile,
    NodeTypeProfile,
    PropertyProfile,
    RelationshipTypeProfile,
)

## 1. The minimal profile — just counts

The smallest valid `GraphProfile` carries a `source` identifier and whatever
node/relationship types were found.  `source` is a free-form string — a
connection URI, an environment name, or a snapshot label.

Even at this level the profile is useful: you can tell which labels exist, how
many nodes there are, and whether any relationship types are present.

In [2]:
minimal = GraphProfile(
    source="neo4j://prod:7687",
    node_type_profiles={
        "Person": NodeTypeProfile(label="Person", count=120),
        "Movie": NodeTypeProfile(label="Movie", count=38),
        "City": NodeTypeProfile(label="City", count=12),
    },
    rel_type_profiles={
        "ACTED_IN": RelationshipTypeProfile(
            rel_type="ACTED_IN",
            count=253,
            source_labels={"Person"},
            target_labels={"Movie"},
        ),
        "LIVES_IN": RelationshipTypeProfile(
            rel_type="LIVES_IN",
            count=90,
            source_labels={"Person"},
            target_labels={"City"},
        ),
    },
)

print("source           :", minimal.source)
print("node labels      :", sorted(minimal.node_labels))
print("relationship types:", sorted(minimal.relationship_types))
print("Person count     :", minimal.node_type_profiles["Person"].count)
print(
    "ACTED_IN endpoints:",
    minimal.rel_type_profiles["ACTED_IN"].source_labels,
    "->",
    minimal.rel_type_profiles["ACTED_IN"].target_labels,
)

source           : neo4j://prod:7687
node labels      : ['City', 'Movie', 'Person']
relationship types: ['ACTED_IN', 'LIVES_IN']
Person count     : 120
ACTED_IN endpoints: {'Person'} -> {'Movie'}


## 2. Adding property profiles — completeness and types

A `PropertyProfile` records three key facts about a property:

1. **`present_count` / `total_count`** — how many nodes had this property set
   to a non-null value out of all nodes of this type.
2. **`completeness`** (computed) — `present_count / total_count`, a float in
   `[0.0, 1.0]`.  `1.0` means every node has the property; `0.0` means none do.
3. **`observed_types`** — the database type names actually seen (e.g. `'String'`,
   `'Long'`).  More than one type in this list means the property is
   heterogeneous — a common data-quality signal.

> **ADR-034 §5 — completeness vs. is_required:**  
> The old `is_required: bool` field has been removed.  Presence is now expressed
> as *observed completeness* (a continuous measure, 0–1) rather than a binary
> flag.  Whether the database enforces presence is captured separately in
> `constraint_required` (§5 below).

In [3]:
with_props = GraphProfile(
    source="neo4j://prod:7687",
    node_type_profiles={
        "Person": NodeTypeProfile(
            label="Person",
            count=120,
            property_profiles={
                "name": PropertyProfile(
                    name="name",
                    present_count=120,
                    total_count=120,
                    observed_types=["String"],
                ),
                "born": PropertyProfile(
                    name="born",
                    present_count=90,
                    total_count=120,
                    observed_types=["Long"],
                ),
                "score": PropertyProfile(
                    name="score",
                    # Mixed types — String and Long both observed
                    present_count=30,
                    total_count=120,
                    observed_types=["String", "Long"],
                    observed_type_counts={"String": 25, "Long": 5},
                ),
            },
        ),
    },
)

person = with_props.node_type_profiles["Person"]
for prop_name, pp in person.property_profiles.items():
    print(
        f"  {prop_name:<6}  completeness={pp.completeness:.2f}  "
        f"missing={pp.missing_count:3d}  types={pp.observed_types}"
    )

  name    completeness=1.00  missing=  0  types=['String']
  born    completeness=0.75  missing= 30  types=['Long']
  score   completeness=0.25  missing= 90  types=['String', 'Long']


The `score` property is heterogeneous — two distinct DB types.  In a comparison
against a definition that declares `score: str`, this will produce a
`PROPERTY_TYPE_MISMATCH` issue because `Long` is not `str`.

Notice also:
- `name.completeness = 1.0` — every Person has a name (as expected for a UID property).
- `born.completeness = 0.75` — 25% of persons have no birth year (optional field).
- `score.completeness = 0.25` — a sparse property.

## 3. `BoundedDistribution` — the shared statistical primitive (ADR-034 §3)

`BoundedDistribution` is the single statistical building block reused across the
profile model for both **value distributions** and **cardinality distributions**.
It stores:

- `count` — number of observations (required).
- `min`, `max`, `mean`, `variance` (and derived `std`) — optional moments.
- `histogram: dict[str, int] | None` — optional per-value breakdown.
- **Truncation honesty**: `sample_complete=False` + `limit` + `other_count` signal
  that the histogram was capped at `limit` top-N values; `other_count` is the
  count of observations not shown.

This primitive prevents the two previously duplicated shapes (`CardinalityStats`
and value distributions) from diverging.  `CardinalityStats` is now a thin
subclass of `BoundedDistribution` — it adds no new fields, just a semantic name.

In [4]:
# --- A complete value distribution (sample_complete=True, no truncation) ---
full_dist = BoundedDistribution(
    count=100,
    min=1.0,
    max=5.0,
    mean=3.2,
    variance=1.76,
    histogram={"1": 10, "2": 20, "3": 35, "4": 25, "5": 10},
    sample_complete=True,
)
print("full distribution")
print(f"  count={full_dist.count}  mean={full_dist.mean}  std={full_dist.std:.4f}")
print(f"  sample_complete={full_dist.sample_complete}  histogram={full_dist.histogram}")
print()

# --- A truncated value distribution (top-3 values kept, 72 others not shown) ---
truncated_dist = BoundedDistribution(
    count=100,
    mean=3.2,
    histogram={"drama": 45, "action": 30, "comedy": 25},  # top 3 only
    sample_complete=False,  # <-- honest: histogram is incomplete
    limit=3,  # top-N cap applied
    other_count=0,  # in this example all 100 observations are in the top 3
)
print("truncated distribution (sample_complete=False)")
print(f"  count={truncated_dist.count}")
print(
    f"  sample_complete={truncated_dist.sample_complete}  limit={truncated_dist.limit}"
)
print(
    f"  shown={sum(truncated_dist.histogram.values())}  other_count={truncated_dist.other_count}"
)

full distribution
  count=100  mean=3.2  std=1.3266
  sample_complete=True  histogram={'1': 10, '2': 20, '3': 35, '4': 25, '5': 10}

truncated distribution (sample_complete=False)
  count=100
  sample_complete=False  limit=3
  shown=100  other_count=0


The `sample_complete=False` flag is critical for honest reporting: a consumer
that only sees `histogram={"drama": 45, "action": 30, "comedy": 25}` might
conclude those are *all* genres.  The flag corrects that assumption immediately.

## 4. Cardinality statistics on relationships

`CardinalityStats` captures the degree distribution of a relationship type from
the source side: for each source node, how many edges of this type does it have?

It is a subclass of `BoundedDistribution`, so `count` here is the number of
source nodes observed (not the total edge count), and `min`/`max`/`mean` are the
per-node degree bounds.

In [5]:
profile_with_card = GraphProfile(
    source="neo4j://prod:7687",
    node_type_profiles={
        "Person": NodeTypeProfile(label="Person", count=120),
        "Movie": NodeTypeProfile(label="Movie", count=38),
    },
    rel_type_profiles={
        "ACTED_IN": RelationshipTypeProfile(
            rel_type="ACTED_IN",
            count=253,
            source_labels={"Person"},
            target_labels={"Movie"},
            # CardinalityStats: per-source-node degree distribution
            # count=120 means 120 Person nodes were assessed
            # min=1 means every assessed Person has at least 1 ACTED_IN edge
            # max=8, mean=2.1
            cardinality_stats=CardinalityStats(
                count=120,
                min=1.0,
                max=8.0,
                mean=2.1,
                variance=3.5,
                histogram={
                    "1": 45,
                    "2": 30,
                    "3": 20,
                    "4": 15,
                    "5": 7,
                    "6": 2,
                    "7": 0,
                    "8": 1,
                },
            ),
        ),
        "DIRECTED": RelationshipTypeProfile(
            rel_type="DIRECTED",
            count=38,
            source_labels={"Person"},
            target_labels={"Movie"},
            # Tighter distribution: exactly one director per movie
            cardinality_stats=CardinalityStats(
                count=30,
                min=1.0,
                max=3.0,
                mean=1.27,
            ),
        ),
    },
)

for rel_type, rtp in profile_with_card.rel_type_profiles.items():
    cs = rtp.cardinality_stats
    if cs is not None:
        print(
            f"  {rel_type:<12}  "
            f"nodes_assessed={cs.count}  "
            f"min={cs.min}  max={cs.max}  mean={cs.mean:.2f}  "
            f"std={cs.std:.3f}"
            if cs.std is not None
            else "std=None"
        )

  ACTED_IN      nodes_assessed=120  min=1.0  max=8.0  mean=2.10  std=1.871
std=None


When `compare_profile_to_definition` (05.02) checks cardinality, it uses
`cardinality_stats.min` and `cardinality_stats.max` against the declared
`CardinalitySpec`.  If either is `None`, the comparison emits
`CARDINALITY_UNVERIFIABLE` (INFO) instead of a hard violation.

## 5. Constraint cross-reference — `constraint_required` (ADR-034 §4)

The `constraint_required` field on `PropertyProfile` is a three-valued signal:

| Value | Meaning |
|---|---|
| `True` | A DB presence/existence constraint was found for this property. |
| `False` | Constraints were inspected; none found for this property. |
| `None` | Constraint information is unavailable for this backend/strategy (ADR-033). |

When `None`, the comparison engine emits `CONSTRAINT_UNVERIFIABLE` (INFO) rather
than silently skipping the check.  `False` is a meaningful signal: the DB was
inspected and no constraint guards this property — even if the profile shows
`completeness=1.0`, that completeness might be accidental.

Below: a profile where `name` is constraint-guaranteed, `born` has no constraint,
and `score` was not inspected.

In [6]:
prof_constraints = GraphProfile(
    source="neo4j://prod:7687",
    node_type_profiles={
        "Person": NodeTypeProfile(
            label="Person",
            count=120,
            property_profiles={
                "name": PropertyProfile(
                    name="name",
                    present_count=120,
                    total_count=120,
                    observed_types=["String"],
                    constraint_required=True,  # DB constraint found
                ),
                "born": PropertyProfile(
                    name="born",
                    present_count=90,
                    total_count=120,
                    observed_types=["Long"],
                    constraint_required=False,  # inspected, no constraint
                ),
                "score": PropertyProfile(
                    name="score",
                    present_count=30,
                    total_count=120,
                    observed_types=["Long"],
                    constraint_required=None,  # not inspected (e.g. NetworkX backend)
                ),
            },
        ),
    },
    # ConstraintInfo records the raw DB constraint metadata
    constraints=[
        ConstraintInfo(
            name="person_name_exists",
            constraint_type="NODE_PROPERTY_EXISTENCE",
            entity_type="NODE",
            labels=["Person"],
            properties=["name"],
        )
    ],
)

person = prof_constraints.node_type_profiles["Person"]
for prop_name, pp in person.property_profiles.items():
    cr = pp.constraint_required
    label = {
        True: "guaranteed by DB",
        False: "no constraint found",
        None: "unavailable",
    }[cr]
    print(f"  {prop_name:<6}  constraint_required={str(cr):<5}  ({label})")

print()
print("Raw constraints in profile:")
for c in prof_constraints.constraints:
    print(
        f"  {c.name}  type={c.constraint_type}  labels={c.labels}  props={c.properties}"
    )

  name    constraint_required=True   (guaranteed by DB)
  born    constraint_required=False  (no constraint found)
  score   constraint_required=None   (unavailable)

Raw constraints in profile:
  person_name_exists  type=NODE_PROPERTY_EXISTENCE  labels=['Person']  props=['name']


## 6. Value distributions and truncation honesty

A `PropertyProfile` can carry a `value_distribution: BoundedDistribution | None`
that records the top-N observed values for that property.  This is useful for
detecting enum-like properties and for the `PropertyEnumValueRule` (E45).

When the backend capped the scan at N distinct values, `sample_complete=False`
and `other_count` reports how many observations fell outside the shown histogram.

In [7]:
# A property whose values are effectively an enum — all 120 distinct values captured.
genre_dist_complete = BoundedDistribution(
    count=120,
    histogram={"drama": 45, "action": 42, "comedy": 33},
    sample_complete=True,
)

# A free-text property — 4 000 distinct values, only top-5 shown.
title_dist_truncated = BoundedDistribution(
    count=4000,
    histogram={
        "The Matrix": 3,
        "Inception": 2,
        "Avatar": 2,
        "Heat": 1,
        "Dune": 1,
    },
    sample_complete=False,
    limit=5,
    other_count=3991,  # 3991 observations in values not in the top-5
)

prof_with_vdist = GraphProfile(
    source="demo",
    node_type_profiles={
        "Movie": NodeTypeProfile(
            label="Movie",
            count=4000,
            property_profiles={
                "genre": PropertyProfile(
                    name="genre",
                    present_count=120,
                    total_count=120,
                    observed_types=["String"],
                    value_distribution=genre_dist_complete,
                ),
                "title": PropertyProfile(
                    name="title",
                    present_count=4000,
                    total_count=4000,
                    observed_types=["String"],
                    value_distribution=title_dist_truncated,
                ),
            },
        ),
    },
)

for prop_name, pp in prof_with_vdist.node_type_profiles[
    "Movie"
].property_profiles.items():
    vd = pp.value_distribution
    if vd is None:
        print(f"  {prop_name}: no distribution")
    else:
        shown = sum(vd.histogram.values()) if vd.histogram else 0
        print(
            f"  {prop_name:<6}  complete={vd.sample_complete}  "
            f"shown={shown}  other={vd.other_count}  top={list(vd.histogram.items())[:3] if vd.histogram else None}"
        )

  genre   complete=True  shown=120  other=0  top=[('drama', 45), ('action', 42), ('comedy', 33)]
  title   complete=False  shown=9  other=3991  top=[('The Matrix', 3), ('Inception', 2), ('Avatar', 2)]


## 7. Serialising and reloading a profile (JSON round-trip)

A `GraphProfile` is a Pydantic model.  It serialises cleanly to JSON and
round-trips without loss.  This is how profiles are persisted (e.g. to
`scratch/profile_export.json`) and reloaded for offline comparison.

The only subtlety is the `timestamp` field: `datetime.now()` is set at
construction time by default.  When reloading, Pydantic parses the ISO-format
string back to a `datetime`, so the round-trip is exact.

In [8]:
# Use the profile built in §4 (has cardinality stats)
original = profile_with_card

# --- Serialise ---
json_str = original.model_dump_json(indent=2)
print("JSON (first 400 chars):")
print(json_str[:400], "...")
print()

# --- Reload ---
reloaded = GraphProfile.model_validate_json(json_str)

# --- Verify round-trip equality ---
# frozen Pydantic models support == by field equality
assert reloaded.source == original.source
assert reloaded.node_labels == original.node_labels
assert reloaded.relationship_types == original.relationship_types

orig_cs = original.rel_type_profiles["ACTED_IN"].cardinality_stats
reload_cs = reloaded.rel_type_profiles["ACTED_IN"].cardinality_stats
assert orig_cs.count == reload_cs.count
assert orig_cs.mean == reload_cs.mean

print("Round-trip OK — reloaded profile matches original.")
print(f"Reloaded source     : {reloaded.source}")
print(f"Reloaded node labels: {sorted(reloaded.node_labels)}")

JSON (first 400 chars):
{
  "source": "neo4j://prod:7687",
  "timestamp": "2026-06-22T15:00:29.341444",
  "node_type_profiles": {
    "Person": {
      "label": "Person",
      "count": 120,
      "property_profiles": {}
    },
    "Movie": {
      "label": "Movie",
      "count": 38,
      "property_profiles": {}
    }
  },
  "rel_type_profiles": {
    "ACTED_IN": {
      "rel_type": "ACTED_IN",
      "count": 253,
     ...

Round-trip OK — reloaded profile matches original.
Reloaded source     : neo4j://prod:7687
Reloaded node labels: ['Movie', 'Person']


## 8. Rendering a profile as text

`orthograph.api.visualization.render_profile(profile)` formats the profile as a
human-readable text table — the same output that appears in CI logs or notebook
cells when you inspect a live database.

In [9]:
# Build a richer profile for display
rich_profile = GraphProfile(
    source="neo4j://prod:7687",
    node_type_profiles={
        "Person": NodeTypeProfile(
            label="Person",
            count=120,
            property_profiles={
                "name": PropertyProfile(
                    name="name",
                    present_count=120,
                    total_count=120,
                    observed_types=["String"],
                    constraint_required=True,
                ),
                "born": PropertyProfile(
                    name="born",
                    present_count=90,
                    total_count=120,
                    observed_types=["Long"],
                    constraint_required=False,
                ),
            },
        ),
        "Movie": NodeTypeProfile(
            label="Movie",
            count=38,
            property_profiles={
                "title": PropertyProfile(
                    name="title",
                    present_count=38,
                    total_count=38,
                    observed_types=["String"],
                    constraint_required=True,
                ),
                "released": PropertyProfile(
                    name="released",
                    present_count=38,
                    total_count=38,
                    observed_types=["Long"],
                ),
            },
        ),
    },
    rel_type_profiles={
        "ACTED_IN": RelationshipTypeProfile(
            rel_type="ACTED_IN",
            count=253,
            source_labels={"Person"},
            target_labels={"Movie"},
            property_profiles={
                "role": PropertyProfile(
                    name="role",
                    present_count=253,
                    total_count=253,
                    observed_types=["String"],
                ),
            },
            cardinality_stats=CardinalityStats(
                count=120, min=1.0, max=8.0, mean=2.1, variance=3.5
            ),
        ),
    },
)

print(visualization.render_profile(rich_profile))

Profile: neo4j://prod:7687
Timestamp: 2026-06-22 15:00:29.579843

Node Types
------------------------------------------------------------
  Person (120 instances)
    name: 100% complete (120/120) [constrained] types=[String]
    born: 75% complete (90/120) [unconstrained] types=[Long]

  Movie (38 instances)
    title: 100% complete (38/38) [constrained] types=[String]
    released: 100% complete (38/38) types=[Long]

Relationship Types
------------------------------------------------------------
  ACTED_IN (253 instances)
    sources: ['Person']
    targets: ['Movie']
    cardinality: min=1.0, max=8.0, avg=2.1, sample_size=120
    role: 100% complete (253/253) types=[String]

